# TCGA-BRCA: tümör ve normal doku arasında diferansiyel ekspresyon
## Batch etkisi ve yöntem seçiminin yanlış pozitiflere etkisi

Bu notebook projenin tüm aşamalarını `src/` içindeki fonksiyonlarla baştan sona çalıştırır.

- **Hızlı adımlar yeniden hesaplanır:** veri yükleme, ön işleme, rank testleri, benchmark ve zenginleştirme.
- **Ağır adımlar diskten yüklenir:** PyDESeq2, ComBat-seq, 2216 permütasyon çalışması ve iç içe çapraz doğrulama. Bu sonuçlar `src/de_methods.py`, `src/permutation.py` ve `src/downstream.py` ile üretildi; nasıl yeniden üretileceği README'de anlatılıyor.
- **Tekrarlanabilirlik kontrolü:** Yeniden hesaplanan sonuçlar, diskteki kayıtlı sonuçlarla karşılaştırılır.

Tüm rastgele işlemlerde seed = 42 kullanıldı.

In [1]:
import sys, json, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

import load, batch, de_methods as dm, permutation as pm, benchmark as bm, downstream as ds, plots, report

TABLES = ROOT / "results" / "tables"
RERUN_HEAVY = False  # True: rerun PyDESeq2 / ComBat-seq for Stage 3 (~25 min)
print(sys.executable)

c:\Users\gundo\Desktop\SPECİAL PROJECT\Gene-Expression-Analysis-of-Tumor-and-Normal-Tissue-in-Breast-Cancer-TCGA-BRCA\.venv\Scripts\python.exe


## Aşama 1 — Veri yükleme ve etiketleme

GDC'den indirilen 1231 STAR "augmented gene counts" dosyasından `unstranded` sütunu okunur ve gen × örnek tamsayı matrisi oluşturulur. Baştaki `N_*` özet satırları atılır.

Aliquot barkodları `metadata.repository*.json` dosyasından alınır. Barkoddan şu bilgiler çıkarılır: TSS, hasta, örnek tipi, plate ve center.

Analizde sadece primer tümör (01) ve solid tissue normal (11) örnekleri kalır. Bir hastanın birden fazla tümör aliquot'u varsa alfabetik olarak ilk olanı tutulur.

In [2]:
load.main()
samples = pd.read_csv(ROOT / "data/processed/samples.csv")
samples["condition"].value_counts()

counts: 60660 genes x 1231 files
n_files: 1231
sheet_rows: 1231
sheet_sample_id_mismatch: 0
sheet_case_id_mismatch: 0
sample_type_counts_all: {'01': 1111, '06': 7, '11': 113}
centers: {'07': 1231}
patients_multi_tumor: 11
patients_multi_normal: 0
dropped_duplicates: ['TCGA-A7-A0DB-01A-11R-A277-07', 'TCGA-A7-A0DB-01C-02R-A277-07', 'TCGA-A7-A0DC-01B-04R-A22O-07', 'TCGA-A7-A13D-01A-13R-A277-07', 'TCGA-A7-A13D-01B-04R-A277-07', 'TCGA-A7-A13E-01A-11R-A277-07', 'TCGA-A7-A13E-01B-06R-A277-07', 'TCGA-A7-A13G-01B-04R-A22O-07', 'TCGA-A7-A26E-01A-11R-A277-07', 'TCGA-A7-A26E-01B-06R-A277-07', 'TCGA-A7-A26F-01B-04R-A22O-07', 'TCGA-A7-A26I-01B-06R-A22O-07', 'TCGA-A7-A26J-01A-11R-A277-07', 'TCGA-A7-A26J-01B-02R-A277-07', 'TCGA-AC-A2QH-01B-04R-A22O-07', 'TCGA-AC-A3OD-01B-06R-A22O-07']
clinical_conflicts: {'age': 0, 'stage': 21, 'race': 0, 'vital_status': 0}
patients_without_clinical: 0
n_tumor: 1095
n_normal: 113
n_paired_patients: 113
n_same_plate_paired_patients: 89
n_plates: 44
n_plates_with_normal

condition
tumor     1095
normal     113
Name: count, dtype: int64

**Yorum.** Sonuçlar beklenen sayılarla örtüşüyor:
- Dosyaların dağılımı: 1111 primer tümör, 113 normal ve 7 metastaz (metastazlar çıkarıldı).
- 11 hastanın birden fazla tümör aliquot'u vardı. Tekrarlar atılınca 1095 tümör ve 113 normal kaldı.
- 113 hastanın hem tümör hem normal örneği var. Bunların 89'unda iki örnek aynı plate'te.
- Plate sayısı tekrarlar atılmadan önce 45, sonra 44. A277 plate'inde sadece tekrar aliquot'lar vardı.
- Center tüm örneklerde 07 olduğu için batch değişkeni olarak kullanılmıyor.

## Aşama 2 — Ön işleme ve batch tanılaması

- **Gen filtresi:** Protein-coding genler alınır. Bir gen, en az 113 örnekte (normal örnek sayısı) CPM ≥ 1 ise tutulur.
- **Normalizasyon:** DESeq2 median-of-ratios size factor'leri kullanılır.
- **Aykırı örnekler:** PCA ve kütüphane boyutuna göre işaretlenir ama silinmez.
- **Batch etkisi:** İlk 10 PC'nin ne kadarını plate, TSS ve condition'ın açıkladığı R² ile ölçülür.
- **Plate gruplama:** 10'dan az örnekli plate'ler, işlem sırasına göre komşu plate'lerle birleştirilir; sonuç `batch_group` sütunudur.

In [3]:
res2 = batch.main()
plots.make_stage2_figures(res2)
print("genes kept:", res2["filt"].shape[0], "| outliers flagged:", int(res2["flags"]["outlier_flag"].sum()))
t = res2["test_plate"]
print(f"plate x condition: chi2={t['chi2']:.1f}, p={t['p']:.2g}, Cramer's V={t['cramers_v']:.3f}")
print("batch groups:", len(res2["batch_table"]), "| with both conditions:", len(res2["both"]),
      "| samples in them:", int(res2["samples"]["batch_has_both"].sum()))
res2["pcv"].round(3)

genes kept: 15317 | outliers flagged: 15
plate x condition: chi2=368.1, p=3.1e-53, Cramer's V=0.552
batch groups: 38 | with both conditions: 15 | samples in them: 615


,PC,var_explained,R2_plate,adjR2_plate,R2_tss,adjR2_tss,R2_condition,adjR2_condition
0,PC1,0.129,0.208,0.178,0.180,0.153,0.503,0.502
1,PC2,0.110,0.073,0.038,0.043,0.011,0.005,0.005
2,PC3,0.068,0.505,0.486,0.234,0.208,0.017,0.016
3,PC4,0.045,0.094,0.061,0.076,0.045,0.160,0.159
4,PC5,0.040,0.144,0.112,0.077,0.046,0.005,0.004
5,PC6,0.028,0.039,0.004,0.052,0.020,0.008,0.008
6,PC7,0.022,0.070,0.036,0.055,0.024,0.008,0.008
7,PC8,0.017,0.051,0.015,0.057,0.026,0.014,0.014
8,PC9,0.014,0.084,0.050,0.043,0.011,0.000,-0.001
9,PC10,0.014,0.126,0.093,0.093,0.063,0.001,0.000


![Fig 2](../results/figures/fig2_pca.png)

![Fig 3](../results/figures/fig3_plate_condition_heatmap.png)

**Yorum.**
- **PC1:** Tümör ile normali ayırıyor (condition R² = 0.50).
- **PC3:** Neredeyse yarısını plate açıklıyor (R² = 0.50). Bu etki işlem sırasına bağlı: geç plate'ler sadece tümör içeriyor ve PC3'te pozitif tarafta (`figS_pc3_by_plate.png`).
- **Plate ile condition güçlü şekilde iç içe:** Cramér's V = 0.55.
  - A089 plate'inde 22 normal var, hiç tümör yok.
  - 27 plate'te sadece tümör var.
  - 38 batch grubunun yalnızca 15'i iki koşulu birden içeriyor. Bu 15 grupta 615 örnek var.

## Aşama 3 — Yedi DE protokolü

| | Yöntem |
|---|---|
| P1 | PyDESeq2 `~condition` (apeGLM ile shrink edilmiş log2FC) |
| P2 | PyDESeq2 `~batch_group + condition` |
| P3 | ComBat-seq (R sva), ardından PyDESeq2 `~condition` |
| P4 | Normalize count'larda Wilcoxon rank-sum |
| P5 | ComBat-seq ile düzeltilmiş count'larda Wilcoxon |
| P6 | van Elteren stratifiye Wilcoxon (strata = `batch_group`) |
| P7 | Koşul içi 95. persentilde winsorize edilmiş count'larla PyDESeq2 |

DEG tanımı: FDR (Benjamini-Hochberg) < 0.05 ve |log2FC| > 1.

Önce van Elteren testinin doğrulama testleri çalıştırılır. Ardından hızlı protokoller (P4, P6) yeniden hesaplanıp kayıtlı tablolarla karşılaştırılır.

In [4]:
out = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")], capture_output=True, text=True)
print(out.stdout[-300:])

...                                                                      [100%]
3 passed in 1.32s



In [5]:
counts, samples_i, genes = dm.load_data()
check = {}
for p in ["P4", "P6"]:
    new, _ = dm.run_protocol(p, counts, samples_i)
    old = dm.load_protocol(p)
    check[p] = {"same DEG set": set(new.index[new.is_DEG]) == set(old.index[old.is_DEG]),
                "max |p diff|": float(np.nanmax(np.abs(new["pvalue"].to_numpy() - old["pvalue"].to_numpy())))}
if RERUN_HEAVY:
    dm.main(["P1", "P2", "P3", "P5", "P7", "--force"])
pd.DataFrame(check).T

,same DEG set,max |p diff|
P4,True,0.0
P6,True,0.0


In [6]:
r3 = dm.summarize_protocols()
plots.fig4_deg_counts(r3["info"]); plots.fig_deg_jaccard(r3["jaccard"])
display(r3["info"][["n_used", "n_DEG", "n_up", "n_down", "seconds"]].round(1))
common = dm.common_lfc_sensitivity()
display(Markdown("**DEG Jaccard: own log2FC vs common log2FC**"))
display(pd.concat({"own": r3["jaccard"].round(2), "common": common["jaccard"].round(2)}, axis=1))

,n_used,n_DEG,n_up,n_down,seconds
protocol,,,,,
P1,1208,3734,2108,1626,32.8
P2,615,3711,1906,1805,70.8
P3,1208,3804,1762,2042,738.3
P4,1208,3570,1160,2410,4.9
P5,1208,3590,1082,2508,712.4
P6,615,3563,1160,2403,3.1
P7,1208,3762,1876,1886,29.5


**DEG Jaccard: own log2FC vs common log2FC**

own                                     common                                    
      P1    P2    P3    P4    P5    P6    P7     P1    P2    P3    P4    P5    P6    P7
P1  1.00  0.82  0.78  0.59  0.56  0.59  0.84   1.00  0.82  0.78  0.94  0.74  0.79  0.99
P2  0.82  1.00  0.89  0.64  0.63  0.64  0.83   0.82  1.00  0.89  0.79  0.85  0.96  0.82
P3  0.78  0.89  1.00  0.65  0.65  0.65  0.81   0.78  0.89  1.00  0.77  0.95  0.86  0.78
P4  0.59  0.64  0.65  1.00  0.87  1.00  0.68   0.94  0.79  0.77  1.00  0.77  0.81  0.94
P5  0.56  0.63  0.65  0.87  1.00  0.87  0.64   0.74  0.85  0.95  0.77  1.00  0.88  0.75
P6  0.59  0.64  0.65  1.00  0.87  1.00  0.68   0.79  0.96  0.86  0.81  0.88  1.00  0.79
P7  0.84  0.83  0.81  0.68  0.64  0.68  1.00   0.99  0.82  0.78  0.94  0.75  0.79  1.00

![Fig 4](../results/figures/fig4_deg_counts.png)

**Yorum.**
- **Markerlar:** Yedi protokolün hepsi beş kanonik markerı doğru yönde buluyor.
- **DEG sayısını log2FC eşiği belirliyor:** 1208 örnekle genlerin %80'inden fazlası FDR < 0.05 çıkıyor.
- **Ortak log2FC'nin etkisi:** Kendi log2FC'leriyle DESeq2 ve rank aileleri belirgin şekilde ayrışıyor (Jaccard yaklaşık 0.6). Ortak bir DESeq2 log2FC kullanılınca P1 ile P4 arasındaki Jaccard 0.59'dan 0.94'e çıkıyor.
- **Sonuç:** Ailelerin ayrışmasının çoğu testten değil, log2FC'nin ortalama ya da medyanla tanımlanmasından geliyor.
- **P7 notu:** P7 başta tüm örnekler üzerinden winsorize edilmişti. Bu, %9'luk normal grubu kırptığı için ADH1B'nin log2FC'sini −3.6'ya düşürdü; o yüzden winsorization koşul içinde yapıldı.

## Aşama 4 — Permütasyon analizi (projenin özgün katkısı)

Bu bölüm 2216 protokol çalışmasının diskteki sonuçlarını yükler.

- **Null A (yöntem kaynaklı):** Etiketler tüm örnekler arasında karıştırılır.
- **Null B (batch kaynaklı):** Sadece tümör örnekleri kullanılır, dolayısıyla bulunan her DEG yanlış pozitiftir. Sahte normaller iki şekilde seçilir:
  - **B-rand:** Rastgele seçilir.
  - **B-conf:** Gerçek verideki plate yapısını taklit eder: yaklaşık %20'si tamamen sahte-normal plate'lerde, plate × etiket Cramér's V değeri gerçek veriye yakın.
  - **FP_batch** = FP_conf − FP_rand. İkisi aynı replikasyonda, aynı örnekler ve aynı sayıda sahte normalle hesaplanır.
- **Null C (ikincil):** Etiketler sadece iki koşulu da içeren 15 batch grubu içinde karıştırılır.
- **Hangi veri:** Rank testleri (P4, P6) tam veride B = 100 ile çalışır. Diğer protokoller alt örneklemde B = 50 ile çalışır: Null A'da 113 normal + 339 tümör, Null B'de 452 tümör.

In [7]:
r4 = pm.run_summary()
cols = ["protocol", "FP_A", "FP_rand", "FP_conf", "FP_batch", "FP_batch_ci_lo", "FP_batch_ci_hi", "FP_C", "FDR_A"]
for thr, d in r4["summary"].groupby("threshold", sort=False):
    display(Markdown(f"**{thr}**")); display(d[cols].round(2).set_index("protocol"))
display(pd.read_csv(TABLES / "perm_label_diagnostics.csv", index_col=0).round(3))

**FDR<0.05**

,FP_A,FP_rand,FP_conf,FP_batch,FP_batch_ci_lo,FP_batch_ci_hi,FP_C,FDR_A
protocol,,,,,,,,
P1,356.68,441.12,647.20,206.08,13.99,423.77,653.06,0.03
P2,90.54,195.48,154.74,-40.74,-119.70,31.02,82.34,0.01
P3,240.22,256.72,334.32,77.60,-8.91,165.57,208.52,0.02
P4,0.03,0.04,378.36,378.32,208.84,578.18,43.82,0.00
P5,0.74,11.04,384.60,373.56,300.82,454.90,5.62,0.00
P6,0.06,0.04,0.27,0.23,-0.05,0.72,0.18,0.00
P7,346.56,378.84,717.94,339.10,87.00,616.66,813.38,0.03


**FDR<0.05 & |log2FC|>1**

,FP_A,FP_rand,FP_conf,FP_batch,FP_batch_ci_lo,FP_batch_ci_hi,FP_C,FDR_A
protocol,,,,,,,,
P1,58.46,115.76,107.64,-8.12,-27.22,9.66,86.28,0.02
P2,12.72,55.08,48.76,-6.32,-20.42,6.66,17.64,0.00
P3,40.92,80.50,77.52,-2.98,-19.94,13.54,34.46,0.01
P4,0.00,0.00,1.96,1.96,0.96,3.12,3.18,0.00
P5,0.54,4.46,240.34,235.88,196.86,278.38,0.22,0.00
P6,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00
P7,34.64,102.22,97.76,-4.46,-21.60,11.72,70.40,0.01


,n,fake_normal,V_mean,V_sd,corr_with_truth_mean,corr_with_truth_sd
null,,,,,,
A_full,1208,113.0,0.190,0.024,-0.001,0.030
A_sub,452,113.0,0.303,0.029,-0.007,0.047
Bconf_full,1095,102.0,0.567,0.012,NaN,NaN
Bconf_sub,452,42.0,0.630,0.018,NaN,NaN
Brand_full,1095,102.0,0.191,0.026,NaN,NaN
Brand_sub,452,42.0,0.286,0.032,NaN,NaN
C_full,615,91.0,0.261,0.007,0.068,0.042
C_sub,252,91.0,0.336,0.005,0.109,0.067


![Fig 5](../results/figures/fig5_false_positives.png)

**Yorum.**
- **Li et al. 2022 tekrarlandı:** DESeq2 tabanlı protokoller, etiketler rastgele karıştırıldığında bile replikasyon başına yüzlerce yanlış pozitif veriyor (P1: 357). Rank testleri neredeyse sıfır.
- **Ama tahmini FDR düşük:** Gerçek veride yaklaşık 12 bin gen anlamlı olduğu için tahmini FDR %3'ün altında kalıyor.
- **Batch karışıklığı P4 ve P5'i bozuyor:** Plate'lerle karışık etiketlerde düz Wilcoxon (P4) ortalama 378, ComBat-seq + Wilcoxon (P5) 374 batch kaynaklı yanlış pozitif veriyor.
- **ComBat-seq korumuyor, zarar veriyor:** Etiketler batch'le karışık olduğunda ComBat-seq, koşul kovaryatını korurken batch farkını koşul sinyaline aktarıyor (Nygaard et al. 2016). P5'in bu yanlış pozitiflerinin çoğu |log2FC| > 1 eşiğini de geçiyor (240).
- **van Elteren (P6) her durumda dayanıklı:** FP_batch = 0.2.
- **Null C'ye dikkat:** Etiketler gerçek etiketlerle hâlâ hafifçe ilişkili (korelasyon 0.07–0.11). Yanlış pozitif sayısı bu korelasyonla artıyor; Spearman P1'de 0.83. Yani Null C batch ile biyolojiyi karıştırıyor.

In [8]:
bs = pd.read_csv(TABLES / "batch_sensitive_genes.csv", index_col=0)
pd.DataFrame({p: {"batch-sensitive": int(bs[f"batch_sensitive_{p}"].sum()),
                  "batch-specific": int(bs[f"batch_specific_{p}"].sum())} for p in pm.PROTOCOL_ORDER})

,P1,P2,P3,P4,P5,P6,P7
batch-sensitive,1638,314,601,592,1290,0,2183
batch-specific,817,113,314,592,1290,0,1596


**P7 duyarlılık analizi (global winsorization, "P7g").** Yang et al. 2024'ün prosedürü uygulandı, ama persentil koşul içinde değil tüm örnekler üzerinden hesaplandı. Null A alt örnekleminde 50 replikasyonla çalıştırıldı.

In [9]:
pd.read_csv(TABLES / "p7_winsorization_comparison.csv", index_col=0).round(3)

,FP_A_fdr,ci_lo,ci_hi,median,frac_reps_any,FP_A_fdr_lfc,ci_lo_lfc,ci_hi_lfc,real_sig,real_DEG,FDR_A,FDR_A_lfc,reduction_vs_P1_fdr,reduction_vs_P1_lfc
protocol,,,,,,,,,,,,,,
P1,356.68,289.573,432.384,254.0,1.0,58.46,52.859,64.720,12593,3823,0.028,0.015,0.000,0.000
P7,346.56,251.352,452.936,193.0,1.0,34.64,30.100,39.841,12826,3785,0.027,0.009,0.028,0.407
P7g,21.06,17.260,25.980,17.0,1.0,0.76,0.360,1.320,12679,3401,0.002,0.000,0.941,0.987


**Yorum.**
- **P7g:** Null A'da yanlış pozitifleri P1'e göre %94 azaltıyor. |log2FC| > 1 eşiği eklenince azalma %98.7. Bu, Yang et al.'ın bildirdiği %98.2 ile uyumlu.
- **Koşul içi P7:** Azalma sadece %3.
- **Bedel:** Global winsorization gerçek veride daha az DEG buluyor (3401'e karşı 3785).
- **Ana tablolar:** P7 ana tablolarda koşul içi versiyon olarak kaldı; P7g burada duyarlılık analizi olarak raporlanıyor.

## Aşama 5 — Benchmark

**Referans:** Aynı plate'te eşleşmiş 89 hasta (178 örnek) kullanıldı. Bu çiftlerde tümör ile normal aynı plate'te olduğu için plate karışıklığı yok. İki yöntemle DEG bulundu:
- Hasta içi farklar üzerinde Wilcoxon signed-rank.
- PyDESeq2 `~patient + condition`.

Referans listesi iki yöntemin kesişimi: ikisinde de aynı yönde DEG olan genler.

In [10]:
r5 = bm.run_all()
print(json.dumps(r5["reference_info"], indent=1))
display(pd.concat({"own log2FC": r5["benchmark"][["precision", "recall", "jaccard"]],
                   "common log2FC": r5["benchmark_common_lfc"][["precision", "recall", "jaccard"]]}, axis=1).round(3))

{
 "n_pairs": 89,
 "n_samples": 178,
 "n_plates": 14,
 "wilcoxon_DEG": 3449,
 "deseq_DEG": 3471,
 "both_any_direction": 3262,
 "direction_conflicts": 0,
 "reference_up": 1137,
 "reference_down": 2125,
 "reference_total": 3262,
 "seconds_wilcoxon": 0.26763990009203553,
 "seconds_deseq": 111.242368599982
}


own log2FC                common log2FC               
          precision recall jaccard     precision recall jaccard
protocol                                                       
P1            0.682  0.781   0.572         0.682  0.781   0.572
P2            0.744  0.846   0.656         0.744  0.846   0.656
P3            0.742  0.865   0.665         0.742  0.865   0.665
P4            0.854  0.935   0.806         0.711  0.782   0.594
P5            0.858  0.945   0.817         0.768  0.863   0.685
P6            0.856  0.935   0.807         0.774  0.846   0.679
P7            0.748  0.863   0.668         0.685  0.783   0.576

![Fig 6](../results/figures/fig6_upset.png)

![Fig 7](../results/figures/fig7_markers.png)

**Yorum.**
- **Referans:** 3262 gen (1137 up, 2125 down). İki yöntem arasında yön çelişkisi yok.
- **Kendi log2FC'leriyle:** Rank protokolleri referansa en yakın (Jaccard yaklaşık 0.81).
- **Ortak log2FC ile:** Bu üstünlük kayboluyor; P4 0.59, P1 0.57. Öne çıkanlar batch'i ele alan protokoller: P2, P3, P5 ve P6, Jaccard 0.66–0.69.
- **Neden:** Referansın Wilcoxon yarısı da medyana dayalı log2FC kullanıyor, bu da rank protokollerini kayırıyor.
- **Bağımsızlık sınırı:** Referanstaki 178 örnek protokollerin kullandığı tam veride de var; referans bağımsız bir veri seti değil.
- **Kesişimler (Fig 6):** 7 protokolün ortak kesişimi 2452 gen. Sonraki en büyük kesişimler aileye özgü: 491 gen sadece DESeq2 ailesinde, 395 gen sadece rank ailesinde.

## Aşama 6 — Downstream etki

- **Zenginleştirme:** GO BP 2023 ve KEGG 2021 kütüphaneleri kullanıldı. Arka plan, test edilen 15317 gen. Her protokolün ilk 20 teriminin protokoller arası Jaccard benzerliği hesaplandı.
- **Sınıflandırma:** İç içe (nested) 5 katlı çapraz doğrulama. Gen seçimi her eğitim katmanında yeniden yapılıyor, böylece test katmanından bilgi sızmıyor. Sonuçlar diskten yüklenir.
- **Robust core imzası:** P2, P3, P5 ve P6'nın dördünde de aynı yönde DEG olan ve B-conf replikasyonlarının %5'inden azında anlamlı çıkan genler.

In [11]:
r6 = ds.run_all()
display(r6["enrichment"]["summary"].pivot_table(index="protocol", columns=["library", "direction"], values="n_terms_FDR<0.05"))
display(r6["classification"][["name", "model", "n_genes", "roc_auc", "f1_normal", "stability_selected", "stability_L1-LR", "stability_RF"]].round(4))
for n in ["robust_core", "robust_core_v2"]:
    print(n, json.loads((TABLES / f"{n}_info.json").read_text()))

library   GO_Biological_Process_2023       KEGG_2021_Human      
direction                       down    up            down    up
protocol                                                        
P1                             235.0  61.0            49.0  12.0
P2                             291.0  57.0            48.0   9.0
P3                             264.0  77.0            63.0  11.0
P4                             233.0  68.0            61.0   8.0
P5                             294.0  94.0            57.0   9.0
P6                             214.0  68.0            61.0   8.0
P7                             272.0  48.0            60.0  11.0

,name,model,n_genes,roc_auc,f1_normal,stability_selected,stability_L1-LR,stability_RF
0,P1,L1-LR,3726.8,0.9996,0.9785,0.9159,0.2493,0.4331
1,P1,RF,3726.8,0.9994,0.9559,0.9159,0.2493,0.4331
2,P1_fixed,L1-LR,3734.0,0.9997,0.9223,1.0000,0.1800,0.6005
3,P1_fixed,RF,3734.0,0.9994,0.9563,1.0000,0.1800,0.6005
4,P2,L1-LR,3701.8,0.9996,0.9743,0.9096,0.3100,0.4424
5,P2,RF,3701.8,0.9995,0.9610,0.9096,0.3100,0.4424
6,P4,L1-LR,3576.8,0.9998,0.9704,0.9322,0.2039,0.4420
7,P4,RF,3576.8,0.9994,0.9650,0.9322,0.2039,0.4420
8,P6,L1-LR,3564.4,0.9997,0.9743,0.9313,0.2997,0.4524
9,P6,RF,3564.4,0.9994,0.9650,0.9313,0.2997,0.4524


robust_core {'protocols': ['P2', 'P3', 'P5', 'P6'], 'filter_protocols': ['P2', 'P3', 'P5', 'P6'], 'max_conf_freq': 0.05, 'DEG_all_same_direction': 2628, 'dropped_batch_frequent': 964, 'robust_core': 1664, 'up': 708, 'down': 956}
robust_core_v2 {'protocols': ['P2', 'P3', 'P5', 'P6'], 'filter_protocols': ['P2', 'P3', 'P6'], 'max_conf_freq': 0.05, 'DEG_all_same_direction': 2628, 'dropped_batch_frequent': 618, 'robust_core': 2010, 'up': 810, 'down': 1200}


![Fig 8](../results/figures/fig8_term_jaccard.png)

![Fig 9](../results/figures/fig9_classification.png)

**Yorum.**
- **Zenginleştirme:** Biyoloji tutarlı: up genlerde mitoz ve hücre döngüsü, down genlerde anjiyogenez, kas ve sinir sistemi terimleri. Ama ilk 20 terimin örtüşmesi yöntem ailesine göre değişiyor; GO BP down'da DESeq2 ile rank aileleri arasında Jaccard 0.29–0.43.
- **Sınıflandırma:** Tümör ile normal bu veride çok kolay ayrıldığı için AUC tüm protokollerde ≥ 0.9994. Protokol seçimi sınıflandırma başarısını değiştirmiyor.
- **Gen kararlılığı:** Seçilen DEG listeleri katmanlar arasında kararlı (Jaccard yaklaşık 0.92). L1 modelinin gerçekten kullandığı genler ise değişken (0.18–0.31).
- **Robust core v1:** 1664 gen. Elenen genlerin çoğunu P5'in kendi batch kaynaklı yanlış pozitifleri belirliyor; ADH1B, FABP4 ve LEP bu yüzden düşüyor.
- **Robust core v2:** Sıklık filtresi sadece P2, P3 ve P6'ya uygulanıyor. Sonuç 2010 gen; ADH1B ve FABP4 geri dönüyor.
- **İmza karşılaştırması:** Her iki imza, P1'in tam DEG listesinin yaklaşık yarısı kadar genle aynı başarıyı veriyor. Bu karşılaştırma nested değil, çünkü listeler tüm örneklerle seçildi.

## Aşama 7 — Özet tablo (Tablo 1)

In [12]:
t1 = report.build_table1()
t1.drop(columns=["method"]).round(3).T

protocol,P1,P2,P3,P4,P5,P6,P7
n_samples_informative,1208,615,1208,1208,1208,615,1208
n_DEG,3734,3711,3804,3570,3590,3563,3762
n_up,2108,1906,1762,1160,1082,1160,1876
n_down,1626,1805,2042,2410,2508,2403,1886
FP_A,356.68,90.54,240.22,0.03,0.74,0.06,346.56
FP_rand,441.12,195.48,256.72,0.04,11.04,0.04,378.84
FP_conf,647.2,154.74,334.32,378.36,384.6,0.27,717.94
FP_batch,206.08,-40.74,77.6,378.32,373.56,0.23,339.1
FP_batch_ci_lo,13.994,-119.703,-8.908,208.844,300.82,-0.05,87.0
FP_batch_ci_hi,423.773,31.021,165.572,578.184,454.904,0.72,616.657


## Sonuç

1. **Tasarım batch'le iç içe:** TCGA-BRCA'da plate ile tümör/normal durumu güçlü şekilde iç içe (Cramér's V = 0.55). Örneklerin yarısı tek koşullu batch'lerde.
2. **Rank testleri ve batch:** Rastgele etiketlerde rank testleri yanlış pozitif vermiyor. Ama etiketler plate'lerle karışık olduğunda düz Wilcoxon yüzlerce yanlış pozitif veriyor. Stratifiye van Elteren testi bu karışıklığa karşı tamamen dayanıklı.
3. **ComBat-seq bu tasarımda zararlı:** Koşul kovaryatıyla ComBat-seq uygulamak batch'i düzeltmiyor, aksine batch farkını koşul sinyaline aktarıyor.
4. **DESeq2 ve winsorization:** DESeq2'nin yöntem kaynaklı yanlış pozitifleri gerçek, ama bu veride tahmini FDR %3'ün altında. Global winsorization bu yanlış pozitiflerin %94'ünü gideriyor.
5. **Protokol farkının kaynağı:** Protokoller arasındaki DEG farkının büyük kısmı log2FC tanımından geliyor. Sınıflandırma başarısı protokolden etkilenmiyor; zenginleştirmede ise ilk terimler yöntem ailesine göre değişiyor.